In [1]:
import pandas as pd
import re

file_path = "../exports/crop_statistics_raw_20260607_104708.csv"
df = pd.read_csv(file_path)
print(f"Original shape: {df.shape}")

id_vars = ['id', 'State', 'District', 'Year', 'source_record_hash', 'source_dataset', 'source_resource_id', 'source_system', 'ingested_at']
value_vars = [c for c in df.columns if c not in id_vars]

col_meta = []
seasons = ['Kharif', 'Rabi', 'Summer', 'Whole_Year', 'Winter', 'Autumn']

for col in value_vars:
    match = re.search(r'_(Area|Production|Yield)_', col)
    if match:
        metric_start = match.start()
        crop = col[:metric_start]
        rest = col[metric_start + 1:]
        
        season = 'Unknown'
        for s in seasons:
            if rest.endswith('_' + s):
                season = s
                break
        
        if season != 'Unknown':
            metric = rest[:-(len(season)+1)]
        else:
            metric = rest
            season = 'Whole_Year'
            
        col_meta.append({'original_col': col, 'Crop': crop, 'Metric': metric, 'Season': season})
    else:
        col_meta.append({'original_col': col, 'Crop': 'Unknown', 'Metric': 'Unknown', 'Season': 'Unknown'})

df_meta = pd.DataFrame(col_meta)

unknown_cols = df_meta[df_meta['Metric'] == 'Unknown']['original_col'].tolist()
if unknown_cols:
    print(f"Warning: these columns didn't match the pattern and will be ignored: {unknown_cols[:5]}")
    value_vars = [v for v in value_vars if v not in unknown_cols]
    df_meta = df_meta[df_meta['Metric'] != 'Unknown']

# Reshape the dataset from wide to long format
df_long = pd.melt(df, id_vars=id_vars, value_vars=value_vars, var_name='original_col', value_name='Value')
df_long = df_long.merge(df_meta, on='original_col', how='left')
df_long.drop('original_col', axis=1, inplace=True)

# Pivot the dataset so metrics become columns
df_clean = df_long.pivot_table(
    index=id_vars + ['Crop', 'Season'],
    columns='Metric',
    values='Value',
    aggfunc='first'
).reset_index()

# To keep all minor crops and patterns without losing valid sparse data:
# We only drop rows where ALL the metric columns are missing (NaN).
metric_cols = [c for c in df_clean.columns if c not in id_vars + ['Crop', 'Season']]
df_clean.dropna(subset=metric_cols, how='all', inplace=True)

print(f"Cleaned shape: {df_clean.shape}")

# Clean the Crop column (replace _ with space)
df_clean['Crop'] = df_clean['Crop'].str.replace('_', ' ')

# Save the cleaned data
output_path = "../exports/crop_statistics_cleaned.csv"
df_clean.to_csv(output_path, index=False)
print(f"Saved cleaned data to {output_path}")

df_clean.head()

Original shape: (16312, 1011)
Cleaned shape: (407903, 18)
Saved cleaned data to ../exports/crop_statistics_cleaned.csv


Metric,id,State,District,Year,source_record_hash,source_dataset,source_resource_id,source_system,ingested_at,Crop,Season,Area_Hectare,Production_Bales,Production_Nuts,Production_Tonnes,Yield_Bales_Hectare,Yield_Nuts_Hectare,Yield_Tonne_Hectare
0,55,1. Andaman and Nicobar Islands,1. Nicobars,2000 - 2001,d3bfcbacbfe0ddb4d64d8f9f7b2a602c9380c909240c3b...,"DESAgri Area, Production & Yield",desagri-apy,desagri,2026-06-05 22:24:21.749217,Arecanut,Kharif,1254.0,NaN,NaN,2000.0,NaN,NaN,1.59
1,55,1. Andaman and Nicobar Islands,1. Nicobars,2000 - 2001,d3bfcbacbfe0ddb4d64d8f9f7b2a602c9380c909240c3b...,"DESAgri Area, Production & Yield",desagri-apy,desagri,2026-06-05 22:24:21.749217,Banana,Whole_Year,176.0,NaN,NaN,641.0,NaN,NaN,3.64
2,55,1. Andaman and Nicobar Islands,1. Nicobars,2000 - 2001,d3bfcbacbfe0ddb4d64d8f9f7b2a602c9380c909240c3b...,"DESAgri Area, Production & Yield",desagri-apy,desagri,2026-06-05 22:24:21.749217,Cashewnut,Whole_Year,720.0,NaN,NaN,165.0,NaN,NaN,0.23
3,55,1. Andaman and Nicobar Islands,1. Nicobars,2000 - 2001,d3bfcbacbfe0ddb4d64d8f9f7b2a602c9380c909240c3b...,"DESAgri Area, Production & Yield",desagri-apy,desagri,2026-06-05 22:24:21.749217,Coconut,Whole_Year,18168.0,NaN,65100000.0,NaN,NaN,3583.22,NaN
4,55,1. Andaman and Nicobar Islands,1. Nicobars,2000 - 2001,d3bfcbacbfe0ddb4d64d8f9f7b2a602c9380c909240c3b...,"DESAgri Area, Production & Yield",desagri-apy,desagri,2026-06-05 22:24:21.749217,Ginger,Whole_Year,36.0,NaN,NaN,100.0,NaN,NaN,2.78
